# PND 2025 | Camada GOLD | Dimensões `dim_municipios` e `dim_grupo`

**Objetivo:** construir as duas tabelas de dimensão a partir do dicionário de dados (`.xlsx`) que está na landing,
para descrever os códigos usados na tabela `gold.pnd_2025_proeficiencia`.

| Dimensão | Origem (aba do Excel) | Colunas |
|---|---|---|
| `gold.dim_municipios` | `MUNICÍPIOS` | `co_municipio`, `nm_municipio`, `uf`, **`uf_descritor`** |
| `gold.dim_grupo` | `DICIONÁRIO_DE_VARIÁVEIS` (variável `CO_GRUPO`) | `co_grupo`, `ds_grupo` |

## Por que as dimensões ficam na gold e não na bronze

- A **bronze** guarda o dado *cru, como veio*. Estas dimensões exigem **interpretação e formatação** (localizar o cabeçalho
  em uma planilha com título e notas, separar "código = descrição", padronizar textos, derivar a UF por extenso). Isso é transformação.
- O arquivo original **já está preservado e inalterado** na landing (`dicionarios`) e catalogado em `bronze.ctl_arquivos_apoio`:
  é ele o registro bruto para auditoria.
- Dimensões são **tabelas de consumo**: BI e consultas as cruzam com a fato. Esse é o papel da gold.
- São dados de referência **pequenos** (alguns milhares de linhas): uma camada silver intermediária só acrescentaria etapas.
  Se no futuro surgirem muitas dimensões ou regras de qualidade mais pesadas, uma silver de referência passa a fazer sentido.

## Regras aplicadas
**`dim_municipios`:** localiza o cabeçalho dinamicamente; remove linhas vazias e notas; valida código IBGE de 7 dígitos e UF de 2 letras;
limpa espaços; troca acento agudo/crase usado como apóstrofo (`D´OESTE`) pelo apóstrofo comum (`D'OESTE`);
garante código único; cria `uf_descritor` (nome do estado por extenso) a partir de `uf`.

**`dim_grupo`:** localiza `CO_GRUPO` no dicionário; lê as linhas no formato `código = descrição`; limpa espaços duplicados
e padroniza o hífen (`LETRAS -PORTUGUÊS` vira `LETRAS - PORTUGUÊS`); garante código único.

**Fluxo**
```
Preparação (1-9) → dim_municipios (10-20) → dim_grupo (21-30) → Validação (31-34)
```

> Execute as células **uma a uma, em ordem**. O notebook é reexecutável: as tabelas são recriadas a cada execução.

## Passo 1 | Garantir a biblioteca de leitura de Excel

O pandas usa a biblioteca `openpyxl` para abrir arquivos `.xlsx`. O comando só instala se ela estiver ausente
e não causa dano se já existir. **Deve ser a primeira célula de código**, porque pode reiniciar o Python do notebook.

In [0]:
%pip install -q openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Passo 2 | Definir os parâmetros

- `catalog`: catálogo do projeto.
- `arquivo_dicionario`: nome do `.xlsx` na landing. **Pode ficar vazio:** se houver um único `.xlsx` no volume, ele é usado automaticamente.

In [0]:
dbutils.widgets.text("catalog", "pnd2025", "Catálogo")
dbutils.widgets.text("arquivo_dicionario", "", "Arquivo .xlsx (opcional)")

## Passo 3 | Ler os parâmetros e definir constantes

Concentra nomes de schema, tabelas e caminhos. `AUDIT` coleta métricas ao longo da execução para o resumo final.

In [0]:
CATALOG = dbutils.widgets.get("catalog").lower()
ARQUIVO_XLSX = dbutils.widgets.get("arquivo_dicionario").strip()

PATH_DICIONARIOS = f"/Volumes/{CATALOG}/landing/dicionarios/"
SCHEMA_GOLD = "gold"

TBL_MUNICIPIOS = "dim_municipios"
TBL_GRUPO = "dim_grupo"

AUDIT = {}   # métricas de auditoria da execução


def fq(tabela: str) -> str:
    # nome totalmente qualificado: `catalogo`.gold.tabela
    return f"`{CATALOG}`.{SCHEMA_GOLD}.{tabela}"


print("Catálogo:", CATALOG)
print("Volume  :", PATH_DICIONARIOS)

Catálogo: pnd2025
Volume  : /Volumes/pnd2025/landing/dicionarios/


## Passo 4 | Selecionar o catálogo

As células `%sql` usam nomes sem catálogo (ex.: `gold.dim_grupo`) e dependem deste passo. Se o Python reiniciar, rode-o de novo.

In [0]:
spark.sql(f"USE CATALOG `{CATALOG}`")
display(spark.sql("SELECT current_catalog() AS catalogo_atual"))

catalogo_atual
pnd2025


## Passo 5 | Garantir que o schema gold existe

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Camada gold - tabelas analiticas e agregacoes para consumo'

## Passo 6 | Localizar o arquivo Excel na landing

Se o parâmetro `arquivo_dicionario` estiver vazio, procura `.xlsx` no volume `dicionarios`:
usa o arquivo se houver **um só**; se houver vários, para e lista as opções (preencha o widget com o nome desejado).

In [0]:
import os

disponiveis = sorted(
    f for f in os.listdir(PATH_DICIONARIOS)
    if f.lower().endswith((".xlsx", ".xlsm")) and not f.startswith("~$")
)
print("Arquivos Excel no volume:", disponiveis)

if ARQUIVO_XLSX:
    assert ARQUIVO_XLSX in disponiveis, f"'{ARQUIVO_XLSX}' não encontrado. Disponíveis: {disponiveis}"
elif len(disponiveis) == 1:
    ARQUIVO_XLSX = disponiveis[0]
else:
    raise RuntimeError(f"Informe o widget 'arquivo_dicionario'. Arquivos encontrados: {disponiveis}")

XLSX_PATH = os.path.join(PATH_DICIONARIOS, ARQUIVO_XLSX)
AUDIT["arquivo_origem"] = ARQUIVO_XLSX
print(f"\nUsando: {ARQUIVO_XLSX} ({os.path.getsize(XLSX_PATH) / 1024:.0f} KB)")

Arquivos Excel no volume: ['Dicionário arquivos e variáveis PND 2025 (09-09-2029).xlsx', 'microdados2025_parametros_itens.xlsx']


---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
File <command-6962343925180424>, line 14
     12     ARQUIVO_XLSX = disponiveis[0]
     13 else:
---> 14     raise RuntimeError(f"Informe o widget 'arquivo_dicionario'. Arquivos encontrados: {disponiveis}")
     16 XLSX_PATH = os.path.join(PATH_DICIONARIOS, ARQUIVO_XLSX)
     17 AUDIT["arquivo_origem"] = ARQUIVO_XLSX

RuntimeError: Informe o widget 'arquivo_dicionario'. Arquivos encontrados: ['Dicionário arquivos e variáveis PND 2025 (09-09-2029).xlsx', 'microdados2025_parametros_itens.xlsx']

## Passo 7 | Funções de apoio (normalização e limpeza de texto)

- `norm`: remove acentos, espaços extras e maiúsculas. Serve para **comparar** textos sem depender de grafia
  (ex.: `MUNICÍPIOS` e `municipios` ficam iguais).
- `limpar_texto`: tira espaços nas pontas, troca espaço "duro" (`\xa0`, comum em Excel) por espaço comum e colapsa espaços repetidos.
- `achar_aba`: encontra a aba pelo nome normalizado.

Esta célula só define funções.

In [0]:
import re
import unicodedata

import pandas as pd
from pyspark.sql import functions as F


def norm(texto) -> str:
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", texto).strip().lower()


def limpar_texto(valor):
    if pd.isna(valor):
        return None
    texto = str(valor).replace("\xa0", " ")
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto or None


def achar_aba(nomes_abas, trecho):
    achadas = [a for a in nomes_abas if trecho in norm(a)]
    assert len(achadas) == 1, f"Aba com '{trecho}' não encontrada de forma única. Abas: {nomes_abas}"
    return achadas[0]

## Passo 8 | Abrir o Excel e identificar as abas

Lê só a lista de abas e localiza as duas que interessam, sem depender de posição ou grafia exata do nome.

In [0]:
xls = pd.ExcelFile(XLSX_PATH, engine="openpyxl")
print("Abas do arquivo:", xls.sheet_names)

ABA_MUNICIPIOS = achar_aba(xls.sheet_names, "municipios")
ABA_VARIAVEIS = achar_aba(xls.sheet_names, "variaveis")

print("\nAba de municípios:", ABA_MUNICIPIOS)
print("Aba de variáveis :", ABA_VARIAVEIS)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6962343925180428>, line 1
----> 1 xls = pd.ExcelFile(XLSX_PATH, engine="openpyxl")
      2 print("Abas do arquivo:", xls.sheet_names)
      4 ABA_MUNICIPIOS = achar_aba(xls.sheet_names, "municipios")

NameError: name 'XLSX_PATH' is not defined

In [0]:
## Passo 9 | Tabela de referência de UF (sigla → nome por extenso)

Base da coluna `uf_descritor`. São as **27 unidades da federação**. Esta tabela é fixa e faz parte do código,
porque não existe nas abas do Excel.

In [0]:
UF_DESCRITOR = {
    "AC": "Acre", "AL": "Alagoas", "AP": "Amapá", "AM": "Amazonas", "BA": "Bahia",
    "CE": "Ceará", "DF": "Distrito Federal", "ES": "Espírito Santo", "GO": "Goiás",
    "MA": "Maranhão", "MT": "Mato Grosso", "MS": "Mato Grosso do Sul", "MG": "Minas Gerais",
    "PA": "Pará", "PB": "Paraíba", "PR": "Paraná", "PE": "Pernambuco", "PI": "Piauí",
    "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte", "RS": "Rio Grande do Sul",
    "RO": "Rondônia", "RR": "Roraima", "SC": "Santa Catarina", "SP": "São Paulo",
    "SE": "Sergipe", "TO": "Tocantins",
}
assert len(UF_DESCRITOR) == 27
print(f"{len(UF_DESCRITOR)} UFs mapeadas.")

In [0]:
---
# PARTE A | `dim_municipios`

In [0]:
## Passo 10 | Ler a aba de municípios sem interpretar

Lê a aba **inteira, sem cabeçalho e tudo como texto** (`header=None`, `dtype=str`). Isso é necessário porque a aba tem
um título, uma nota explicativa e só então o cabeçalho real. Interpretar na leitura poderia deslocar colunas.
Veja a amostra: o cabeçalho (`CÓDIGO DO MUNICÍPIO`) não está na primeira linha.

In [0]:
df_raw_mun = xls.parse(ABA_MUNICIPIOS, header=None, dtype=str)

print(f"Dimensões lidas: {df_raw_mun.shape[0]} linhas x {df_raw_mun.shape[1]} colunas")
display(df_raw_mun.head(8))

In [0]:
## Passo 11 | Localizar o cabeçalho e as colunas

Procura a linha que contém `CÓDIGO DO MUNICÍPIO` e, nessa linha, as posições das três colunas necessárias.
Assim o notebook continua funcionando se a planilha ganhar ou perder linhas de título.

In [0]:
linha_cab = None
for i in range(len(df_raw_mun)):
    textos = [norm(v) for v in df_raw_mun.iloc[i] if pd.notna(v)]
    if "codigo do municipio" in textos:
        linha_cab = i
        break

assert linha_cab is not None, "Cabeçalho 'CÓDIGO DO MUNICÍPIO' não encontrado na aba."

posicoes = {norm(v): j for j, v in enumerate(df_raw_mun.iloc[linha_cab]) if pd.notna(v)}
col_cod = posicoes["codigo do municipio"]
col_nome = posicoes["nome do municipio"]
col_uf = posicoes["uf"]

print(f"Cabeçalho na linha {linha_cab + 1} do Excel | colunas (índice): código={col_cod}, nome={col_nome}, uf={col_uf}")

In [0]:
## Passo 12 | Extrair os dados e padronizar o texto

Pega as linhas **abaixo do cabeçalho**, só das três colunas, e aplica a limpeza:

- espaços nas pontas e duplicados removidos;
- código sem sufixo `.0` (o Excel às vezes entrega número como `1100015.0`);
- `uf` em maiúsculas;
- acento agudo, crase e aspas curvas usados como apóstrofo viram apóstrofo comum (`D´OESTE` → `D'OESTE`).

Linhas totalmente vazias são descartadas e contadas.

In [0]:
APOSTROFOS = re.compile(r"[´`’‘]")

df_mun = df_raw_mun.iloc[linha_cab + 1:, [col_cod, col_nome, col_uf]].copy()
df_mun.columns = ["co_municipio", "nm_municipio", "uf"]

df_mun = df_mun.apply(lambda coluna: coluna.map(limpar_texto))
AUDIT["mun_linhas_apos_cabecalho"] = len(df_mun)

df_mun = df_mun.dropna(how="all")                      # remove linhas em branco
AUDIT["mun_linhas_nao_vazias"] = len(df_mun)

df_mun["co_municipio"] = df_mun["co_municipio"].map(lambda v: re.sub(r"\.0+$", "", v) if v else v)
df_mun["uf"] = df_mun["uf"].map(lambda v: v.upper() if v else v)
df_mun["nm_municipio"] = df_mun["nm_municipio"].map(lambda v: APOSTROFOS.sub("'", v) if v else v)

print(f"Linhas após o cabeçalho: {AUDIT['mun_linhas_apos_cabecalho']:,} | não vazias: {AUDIT['mun_linhas_nao_vazias']:,}")

In [0]:
## Passo 13 | Validar as linhas e separar as descartadas

Uma linha só é aceita como município se:

- `co_municipio` tem **exatamente 7 dígitos** (padrão IBGE);
- `uf` tem **2 letras**;
- `nm_municipio` está preenchido.

O que não passar (notas de rodapé, totais, lixo) **não é apagado em silêncio**: vai para `df_descartadas`, que é exibido
para você conferir. Se aparecer algum município de verdade ali, avise: a regra precisa de ajuste.

In [0]:
linha_valida = (
    df_mun["co_municipio"].fillna("").str.fullmatch(r"\d{7}")
    & df_mun["uf"].fillna("").str.fullmatch(r"[A-Z]{2}")
    & df_mun["nm_municipio"].notna()
)

df_descartadas = df_mun[~linha_valida]
df_mun = df_mun[linha_valida].copy()

AUDIT["mun_linhas_descartadas"] = len(df_descartadas)
AUDIT["mun_linhas_validas"] = len(df_mun)

print(f"Válidas: {len(df_mun):,} | descartadas: {len(df_descartadas):,}")
display(df_descartadas)

In [0]:
## Passo 14 | Garantir um registro por município

Remove linhas **totalmente idênticas** (duplicata inofensiva) e depois exige que `co_municipio` seja **único**.
Se um mesmo código aparecer com nome ou UF diferentes, o notebook **para**: não há como escolher qual está certo sem decisão sua.

In [0]:
antes = len(df_mun)
df_mun = df_mun.drop_duplicates()
AUDIT["mun_duplicatas_exatas_removidas"] = antes - len(df_mun)

conflitos = df_mun[df_mun.duplicated("co_municipio", keep=False)]
assert conflitos.empty, f"Código de município repetido com dados diferentes:\n{conflitos}"

print(f"Duplicatas exatas removidas: {AUDIT['mun_duplicatas_exatas_removidas']}")
print(f"Municípios únicos: {len(df_mun):,}")

In [0]:
## Passo 15 | Criar `uf_descritor`

Traduz a sigla (`uf`) para o nome do estado por extenso (`RO` → `Rondônia`), usando a tabela do passo 9.

Se alguma sigla não existir na tabela, o notebook **para** e mostra quais, em vez de gravar nulos silenciosamente.

In [0]:
df_mun["uf_descritor"] = df_mun["uf"].map(UF_DESCRITOR)

ufs_sem_mapa = sorted(df_mun.loc[df_mun["uf_descritor"].isna(), "uf"].unique())
assert not ufs_sem_mapa, f"Siglas de UF sem descrição: {ufs_sem_mapa}"

print("uf_descritor criado para todas as linhas.")
display(df_mun[["uf", "uf_descritor"]].drop_duplicates().sort_values("uf"))

In [0]:
## Passo 16 | Converter para Spark e tipar as colunas

Passa do pandas para um DataFrame Spark. `co_municipio` vira inteiro (o código IBGE não tem zero à esquerda);
as demais ficam como texto.

O pandas foi usado nas etapas anteriores porque a planilha é pequena e a lógica de localizar cabeçalho em células
é mais simples ali. A partir daqui o dado segue em Spark.

**Sobre os nomes:** a planilha de origem traz os municípios **sem acentos** (ex.: `SAO MIGUEL DO GUAPORE`).
A dimensão reproduz a origem; corrigir isso exigiria outra fonte oficial.

In [0]:
colunas_mun = ["co_municipio", "nm_municipio", "uf", "uf_descritor"]

df_dim_mun = (
    spark.createDataFrame(df_mun[colunas_mun])
    .withColumn("co_municipio", F.col("co_municipio").cast("int"))
    .orderBy("co_municipio")
)

df_dim_mun.printSchema()
display(df_dim_mun.limit(10))

In [0]:
## Passo 17 | Funções de gravação, documentação e chave primária

Três funções reaproveitadas pelas duas dimensões:

- `gravar_dimensao`: recria a tabela (`DROP` + `saveAsTable`). O `DROP` evita conflito com restrições de execuções anteriores.
- `comentar_dimensao`: aplica comentários na tabela e nas colunas (visíveis no Catalog Explorer).
- `definir_chave_primaria`: marca as colunas como `NOT NULL` e declara a chave primária. No Unity Catalog ela é
  **informativa** (documenta o modelo e ajuda ferramentas de BI), não é imposta pelo banco: por isso a unicidade é verificada no código.

Esta célula só define funções.

In [0]:
def gravar_dimensao(df, tabela: str) -> str:
    destino = fq(tabela)
    spark.sql(f"DROP TABLE IF EXISTS {destino}")
    df.write.format("delta").mode("overwrite").saveAsTable(destino)
    return destino


def comentar_dimensao(tabela: str, comentario_tabela: str, comentarios_colunas: dict):
    destino = fq(tabela)
    spark.sql(f"COMMENT ON TABLE {destino} IS '{comentario_tabela}'")
    for coluna, texto in comentarios_colunas.items():
        spark.sql(f"COMMENT ON COLUMN {destino}.`{coluna}` IS '{texto}'")


def definir_chave_primaria(tabela: str, colunas: list):
    destino = fq(tabela)
    for c in colunas:
        spark.sql(f"ALTER TABLE {destino} ALTER COLUMN `{c}` SET NOT NULL")
    lista = ", ".join(f"`{c}`" for c in colunas)
    try:
        spark.sql(f"ALTER TABLE {destino} ADD CONSTRAINT pk_{tabela} PRIMARY KEY ({lista})")
        print(f"Chave primária criada: pk_{tabela} ({lista})")
    except Exception as e:
        print(f"Não foi possível criar a chave primária (a tabela segue válida): {str(e)[:200]}")

In [0]:
## Passo 18 | Gravar `gold.dim_municipios`

In [0]:
destino_mun = gravar_dimensao(df_dim_mun, TBL_MUNICIPIOS)
print(f"Tabela gravada: {destino_mun}")

In [0]:
## Passo 19 | Documentar `dim_municipios`

In [0]:
comentar_dimensao(
    TBL_MUNICIPIOS,
    "Dimensao de municipios (codigo IBGE de 7 digitos). Origem: aba MUNICIPIOS do dicionario de dados PND 2025",
    {
        "co_municipio": "Codigo IBGE do municipio (7 digitos: 1o regiao, 1o e 2o UF, 3o ao 6o municipio, 7o digito verificador)",
        "nm_municipio": "Nome do municipio, conforme a planilha de origem",
        "uf": "Sigla da unidade da federacao",
        "uf_descritor": "Nome da unidade da federacao por extenso, derivado da sigla em uf",
    },
)
print("Comentários aplicados.")

In [0]:
## Passo 20 | Definir a chave primária de `dim_municipios`

Chave: `co_municipio`.

In [0]:
definir_chave_primaria(TBL_MUNICIPIOS, ["co_municipio"])

In [0]:
---
# PARTE B | `dim_grupo`

In [0]:
## Passo 21 | Ler a aba do dicionário de variáveis sem interpretar

Mesma estratégia: aba inteira, sem cabeçalho, tudo como texto. O dicionário descreve **todas as variáveis** de todos os arquivos;
as categorias de `CO_GRUPO` (`702 = MATEMÁTICA (LICENCIATURA)`, ...) estão em células ao lado do nome da variável.

In [0]:
df_raw_var = xls.parse(ABA_VARIAVEIS, header=None, dtype=str)

print(f"Dimensões lidas: {df_raw_var.shape[0]} linhas x {df_raw_var.shape[1]} colunas")
display(df_raw_var.head(8))

In [0]:
## Passo 22 | Localizar a variável `CO_GRUPO`

Procura todas as células cujo conteúdo é `CO_GRUPO`. Como o dicionário cobre vários arquivos
(`arq1` a `arq13`) e a variável pode se repetir, **pode haver mais de uma ocorrência**: todas são tratadas
e depois reconciliadas.

In [0]:
ocorrencias = [
    (i, j)
    for i in range(len(df_raw_var))
    for j in range(df_raw_var.shape[1])
    if pd.notna(df_raw_var.iat[i, j]) and norm(df_raw_var.iat[i, j]) == "co_grupo"
]

assert ocorrencias, "Variável CO_GRUPO não encontrada na aba de variáveis."

AUDIT["grupo_ocorrencias_co_grupo"] = len(ocorrencias)
print(f"CO_GRUPO encontrado {len(ocorrencias)} vez(es) (linha, coluna): {ocorrencias}")

In [0]:
## Passo 23 | Coletar as linhas de categoria de cada ocorrência

Na planilha, a descrição de uma variável ocupa **várias linhas** abaixo do nome (células mescladas: o nome só aparece na primeira).
Para cada ocorrência, o bloco vai da linha do nome até a linha **anterior ao próximo nome de variável diferente**.

Dentro do bloco, cada célula é quebrada por linhas (caso a lista venha em uma única célula com quebras) e
guardamos as linhas que seguem o padrão `código = descrição`. O que não segue o padrão (tipo, tamanho, descrição da variável) é ignorado.

In [0]:
PADRAO_CATEGORIA = re.compile(r"^\s*(\d+)\s*=\s*(.+?)\s*$")
LIMITE_LINHAS_BLOCO = 200

registros = []
for linha_ini, col_nome in ocorrencias:
    for r in range(linha_ini, min(linha_ini + LIMITE_LINHAS_BLOCO, len(df_raw_var))):
        celula_nome = df_raw_var.iat[r, col_nome]

        # fim do bloco: apareceu outro nome de variável
        if r > linha_ini and pd.notna(celula_nome) and norm(celula_nome) != "co_grupo":
            break

        for valor in df_raw_var.iloc[r]:
            if pd.isna(valor):
                continue
            for linha in str(valor).splitlines():
                m = PADRAO_CATEGORIA.match(linha)
                if m:
                    registros.append((m.group(1), m.group(2), linha_ini))

print(f"{len(registros)} linha(s) de categoria coletadas (antes de limpar e deduplicar).")

In [0]:
## Passo 24 | Limpar as descrições

Formatação aplicada ao texto de cada grupo:

- espaços duplicados e nas pontas removidos (ex.: `LETRAS  -PORTUGUÊS`);
- hífen padronizado com um espaço de cada lado (`LETRAS -PORTUGUÊS` → `LETRAS - PORTUGUÊS`).

A padronização do hífen é aplicada **só aqui**: nos municípios o hífen faz parte do nome (`GUAJARA-MIRIM`).

In [0]:
def limpar_descricao_grupo(texto: str) -> str:
    texto = re.sub(r"\s+", " ", texto).strip()
    texto = re.sub(r"\s*-\s*", " - ", texto)
    return texto


df_grupo = pd.DataFrame(registros, columns=["co_grupo", "ds_grupo", "linha_origem"])
df_grupo["ds_grupo"] = df_grupo["ds_grupo"].map(limpar_descricao_grupo)

display(df_grupo.drop(columns="linha_origem").drop_duplicates().sort_values("co_grupo", key=lambda s: s.astype(int)))

In [0]:
## Passo 25 | Reconciliar ocorrências e garantir um registro por grupo

Se `CO_GRUPO` aparece em vários arquivos, as listas devem ser **iguais**. Esta etapa:

1. Remove repetições idênticas (mesmo código e mesma descrição).
2. **Para** se o mesmo código tiver descrições diferentes entre ocorrências (conflito real no dicionário).
3. Compara a quantidade com a esperada (`ESPERADOS_GRUPOS = 17`, conforme a lista do dicionário). Divergência gera **aviso**, não erro.

In [0]:
ESPERADOS_GRUPOS = 17

df_grupo = df_grupo.drop(columns="linha_origem").drop_duplicates()

descricoes_por_codigo = df_grupo.groupby("co_grupo")["ds_grupo"].nunique()
conflitos_grupo = descricoes_por_codigo[descricoes_por_codigo > 1]
assert conflitos_grupo.empty, (
    f"Códigos com descrições diferentes entre ocorrências: {list(conflitos_grupo.index)}\n"
    f"{df_grupo[df_grupo['co_grupo'].isin(conflitos_grupo.index)]}"
)

AUDIT["grupo_registros_unicos"] = len(df_grupo)
print(f"Grupos únicos: {len(df_grupo)}")
if len(df_grupo) != ESPERADOS_GRUPOS:
    print(f"ATENÇÃO: esperados {ESPERADOS_GRUPOS} grupos, encontrados {len(df_grupo)}. Confira o passo 24.")

In [0]:
## Passo 26 | Converter `dim_grupo` para Spark e tipar

`co_grupo` vira inteiro; `ds_grupo` fica como texto.

In [0]:
df_dim_grupo = (
    spark.createDataFrame(df_grupo[["co_grupo", "ds_grupo"]])
    .withColumn("co_grupo", F.col("co_grupo").cast("int"))
    .orderBy("co_grupo")
)

df_dim_grupo.printSchema()
display(df_dim_grupo)

In [0]:
## Passo 27 | Gravar `gold.dim_grupo`

Usa a mesma função `gravar_dimensao` do passo 17.

In [0]:
destino_grupo = gravar_dimensao(df_dim_grupo, TBL_GRUPO)
print(f"Tabela gravada: {destino_grupo}")

In [0]:
## Passo 28 | Documentar `dim_grupo`

In [0]:
comentar_dimensao(
    TBL_GRUPO,
    "Dimensao de grupos: area da prova de enquadramento do curso no Enade. Origem: dicionario de variaveis PND 2025 (CO_GRUPO)",
    {
        "co_grupo": "Codigo da area da prova de enquadramento do curso no Enade",
        "ds_grupo": "Descricao da area (curso e grau), conforme o dicionario de dados",
    },
)
print("Comentários aplicados.")

In [0]:
## Passo 29 | Definir a chave primária de `dim_grupo`

Chave: `co_grupo`.

In [0]:
definir_chave_primaria(TBL_GRUPO, ["co_grupo"])

In [0]:
## Passo 30 | Conferir as restrições registradas

Deve listar uma `PRIMARY KEY` para cada dimensão.

In [0]:
%sql
SELECT table_name, constraint_name, constraint_type
FROM information_schema.table_constraints
WHERE table_schema = 'gold'
  AND table_name IN ('dim_municipios', 'dim_grupo')
ORDER BY table_name

In [0]:
---
# VALIDAÇÃO

In [0]:
## Passo 31 | Validar as tabelas gravadas

Relê cada tabela e confirma: linhas gravadas = linhas preparadas, chave **única** e **sem nulos**.
A unicidade é conferida aqui porque a chave primária do Unity Catalog não é imposta.

In [0]:
verificacoes = [
    (TBL_MUNICIPIOS, "co_municipio", df_dim_mun.count()),
    (TBL_GRUPO, "co_grupo", df_dim_grupo.count()),
]

for tabela, chave, esperado in verificacoes:
    df_t = spark.table(fq(tabela))
    total = df_t.count()
    distintas = df_t.select(chave).distinct().count()
    nulas = df_t.where(F.col(chave).isNull()).count()

    AUDIT[f"{tabela}_linhas_gravadas"] = total
    print(f"{tabela}: linhas={total:,} (esperado {esperado:,}) | chaves únicas={distintas:,} | chaves nulas={nulas}")

    assert total == esperado, f"{tabela}: divergência entre linhas preparadas e gravadas."
    assert distintas == total, f"{tabela}: chave duplicada."
    assert nulas == 0, f"{tabela}: chave nula."

print("\nValidação concluída: tudo OK.")

In [0]:
## Passo 32 | Cobertura das dimensões sobre a tabela fato

Verifica se **todos os códigos usados** em `gold.pnd_2025_proeficiencia` existem nas dimensões. A comparação é feita
como texto, para não depender de os tipos serem iguais. Códigos sem correspondência são listados.

Se a tabela fato ainda não existir, a etapa avisa e segue. É uma checagem de qualidade, não uma trava.

In [0]:
def cobertura(fato, col_fato, tabela_dim, col_dim):
    k_fato = fato.select(F.col(col_fato).cast("string").alias("k")).distinct()
    k_dim = spark.table(fq(tabela_dim)).select(F.col(col_dim).cast("string").alias("k"))
    sem_dim = k_fato.join(k_dim, "k", "left_anti")
    return k_fato.count(), sem_dim.count(), sem_dim


try:
    fato = spark.table(fq("pnd_2025_proeficiencia"))

    for col_fato, tabela_dim, col_dim in [
        ("co_municipio_prova", TBL_MUNICIPIOS, "co_municipio"),
        ("co_grupo", TBL_GRUPO, "co_grupo"),
    ]:
        total, faltantes, df_falt = cobertura(fato, col_fato, tabela_dim, col_dim)
        AUDIT[f"cobertura_{col_fato}_sem_dimensao"] = faltantes
        print(f"{col_fato}: {total:,} códigos distintos na fato | sem correspondência em {tabela_dim}: {faltantes:,}")
        if faltantes:
            display(df_falt.limit(20))

except Exception as e:
    print(f"Checagem de cobertura não executada: {str(e)[:200]}")

In [0]:
## Passo 33 | Visualizar `dim_grupo`

In [0]:
%sql
SELECT * FROM gold.dim_grupo ORDER BY co_grupo

In [0]:
## Passo 34 | Visualizar `dim_municipios` por UF

Uma linha por UF com a quantidade de municípios e o `uf_descritor`. Devem aparecer **27 linhas**.

In [0]:
%sql
SELECT uf, uf_descritor, COUNT(*) AS qtd_municipios
FROM gold.dim_municipios
GROUP BY uf, uf_descritor
ORDER BY uf

In [0]:
## Passo 35 | Resumo de auditoria da execução

Métricas coletadas ao longo do notebook: arquivo de origem, linhas lidas, descartadas, gravadas e cobertura sobre a fato.
Para o histórico de versões de cada tabela, use `DESCRIBE HISTORY gold.dim_municipios` e `DESCRIBE HISTORY gold.dim_grupo`.

In [0]:
display(pd.DataFrame([{"metrica": k, "valor": v} for k, v in AUDIT.items()]))

In [0]:
## Próximos passos

- Declarar **chaves estrangeiras informativas** entre `gold.pnd_2025_proeficiencia` e as dimensões, para que ferramentas de BI
  reconheçam o relacionamento automaticamente (exige tipos idênticos nas colunas ligadas).
- Criar uma **view de consumo** unindo a fato às duas dimensões (município, UF por extenso e descrição do grupo).